# Main — Dual-Layer Watermarking Demo App (Gradio, Colab)

Enter a prompt → get **plain**, **layer-1 (public topic) watermarked** and **dual-layer**
continuations from OPT-2.7b, then run BOTH detectors on every output:

- **Layer 1** (`src/detection/topic_detection.py`): re-infers the topic, scores against its
  static uniform greenlist → z-score / p-value / confirmed.
- **Layer 2** (`src/detection/kgw_detection.py`): keyed rolling-greenlist ownership check
  → ownership score / p-value / confirmed.

Everything imports from `src/` — nothing is redefined here. Runs on a Colab GPU runtime
(T4 is fine; ~15–25 s per prompt since we generate 4 × 200 tokens).

**Public link:** the final cell prints a `https://xxxx.gradio.live` URL — anyone can
open the app from it while this Colab session is alive. The link dies when the runtime
disconnects, so re-run the last cell to get a fresh one for each demo.


In [1]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git
%cd Dual_watermarking_Scheme
!pip install -q transformers scipy pandas accelerate gradio

fatal: destination path 'Dual_watermarking_Scheme' already exists and is not an empty directory.
/content/Dual_watermarking_Scheme


In [2]:
import os
import torch
import pandas as pd

from src.utils.model import load_model
from src.utils.loadConfig import load_config
from src.watermark.dual_layer import DualWaterMarking
from src.detection.topic_detection import prepare, detect_topic_watermark
from src.detection.kgw_detection import detect_private_watermark

config = load_config("secondLayer")
print(config)

{'MODEL_NAME': 'facebook/opt-2.7b', 'GREEN_FRACTION': 0.5, 'PREV_TOKEN_SIZE': 5, 'DETECTION_THRESHOLD': 0.6, 'P_VALUE_THRESHOLD': 0.05}


## Load model, generator wrapper and detection state

`DualWaterMarking` resolves the private key from `.env`, or mints one and saves it — the SAME
key object (`wm.key`) is handed to the KGW detector, so generation and detection always agree
within this session. Detection state (green sets + embedding geometry) is built once.

In [3]:
# src/utils/model.py loads to CPU by default -- other notebooks always move it
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

model, tokenizer, VOCAB_SIZE = load_model(config["MODEL_NAME"])
model.to(DEVICE).eval()


wm = DualWaterMarking(
    model,
    tokenizer,
    greenlist_dir="data/greenlist",
    split="all",
    green_fraction=config["GREEN_FRACTION"],
    prev_token_size=config["PREV_TOKEN_SIZE"],
    delta_public=2.0,
    delta_private=0.7,        
    max_new_tokens=200,
    seed=0,
)

state = prepare(model, tokenizer, greenlist_dir="data/greenlist", split="all")

print("topics:", wm.topics)
print("key fingerprint:", wm.key[:8], "...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265
topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']
key fingerprint: 900861d1 ...


## Generate + full detection for one prompt

In [4]:
def analyze(prompt):
    """one prompt -> dict with all three generations and every detector verdict"""

    row = wm.watermark([prompt], include_single_layers=True).iloc[0]

    texts = {
        "plain":   row["plain_output"],
        "layer1_only_output": row["layer1_only_output"],
        "dual_watermarked_output": row["dual_watermarked_output"],
    }


    det1 = {name: detect_topic_watermark(text, tokenizer, state, vocab_size=VOCAB_SIZE)
            for name, text in texts.items()}

   
    det2 = {
        name: detect_private_watermark(
            text=text,
            tokenizer=tokenizer,
            key=wm.key,
            vocab_size=VOCAB_SIZE,
            green_fraction=config["GREEN_FRACTION"],
            prev_token_size=config["PREV_TOKEN_SIZE"],
            threshold=config["DETECTION_THRESHOLD"],
            p_value_threshold=config["P_VALUE_THRESHOLD"],
        )
        for name, text in texts.items()
    }

    return texts, det1, det2, row["topic"]


texts, det1, det2, topic = analyze(
    "Recent advances in artificial intelligence have transformed"
)
print("routed topic:", topic)
for name in texts:
    print(f"\n[{name}] L1 z={det1[name]['z_score']} confirmed={det1[name]['confirmed']}"
          f" | L2 own={det2[name]['ownership_score']} confirmed={det2[name]['confirmed']}")

routed topic: technology

[plain] L1 z=-1.7162 confirmed=False | L2 own=0.4900990099009901 confirmed=False

[layer1_only_output] L1 z=-1.7178 confirmed=False | L2 own=0.4801980198019802 confirmed=False

[dual_watermarked_output] L1 z=7.1248 confirmed=True | L2 own=0.594059405940594 confirmed=False


## Gradio app

Outputs: the three generations, plus a detection report covering the routed topic, layer-1
z-values and verdicts per output, the private-layer ownership check, and the combined
**both-layers** claim (AND rule: topic confirmed AND private confirmed on the dual text).

In [5]:
import gradio as gr

OK = "\U0001F512"      # padlock
BAD = "\u26D4"         # no-entry
CHECK = "\u2705"
CROSS = "\u274C"
ARROW = "\u2192"


def _fmt_verdict(det1_res, det2_res):
    icon1 = CHECK if det1_res["confirmed"] else CROSS
    icon2 = CHECK if det2_res["confirmed"] else BAD
    v1 = "WATERMARKED" if det1_res["confirmed"] else "not confirmed"
    v2 = "OWNED" if det2_res["confirmed"] else "not confirmed"
    return (
        f"| statistic | value |\n"
        f"|---|---|\n"
        f"| L1 detected topic | **{det1_res['topic']}** (cos {det1_res['topic_score']}) |\n"
        f"| L1 z-score | **{det1_res['z_score']}** {ARROW} {icon1} {v1} |\n"
        f"| L1 green hits | {det1_res['match_count']} / {det1_res['num_positions']} "
        f"(gamma {det1_res['gamma']}) |\n"
        f"| L2 ownership score | **{round(det2_res['ownership_score'], 3)}** {ARROW} {icon2} {v2} |\n"
        f"| L2 green hits | {det2_res['match_count']} / {det2_res['num_positions']} |"
    )


def run_app(prompt):
    texts, det1, det2, topic = analyze(prompt)

    plain_md = (
        f"**Routed topic:** `{topic}`\n\n"
        + "**Plain (no watermark)**\n\n"
        + _fmt_verdict(det1["plain"], det2["plain"])
    )

    l1_md = (
        "**Layer-1 only (public topic watermark)**\n\n"
        + _fmt_verdict(det1["layer1_only_output"], det2["layer1_only_output"])
    )

    dual = det1["dual_watermarked_output"]
    dual2 = det2["dual_watermarked_output"]
    both = bool(dual["confirmed"] and dual2["confirmed"])
    claim = OK + " CONFIRMED" if both else BAD + " REJECTED"
    dual_md = (
        "**Dual (both layers)**\n\n"
        + _fmt_verdict(dual, dual2)
        + f"\n\n### Both-layers claim: {claim}"
        + "\n\n(topic AND private ownership must both pass)"
    )

    return texts["plain"], texts["layer1_only_output"], \
           texts["dual_watermarked_output"], plain_md, l1_md, dual_md


with gr.Blocks(title="Dual-Layer Watermarking Demo") as demo:
    gr.Markdown("## 🔐 Dual-Layer Watermarking — OPT-2.7b\n"
                "*Layer 1:* public topic greenlist boost · "
                "*Layer 2:* private keyed KGW boost")

    with gr.Row():
        prompt_in = gr.Textbox(label="Prompt", lines=3,
                               placeholder="e.g. The government announced a new policy on...")
        btn = gr.Button("Generate & Detect", variant="primary")

    with gr.Row():
        out_plain = gr.Textbox(label="Normal response", lines=6, interactive=False)
    with gr.Row():
        out_l1 = gr.Textbox(label="Layer-1 watermarked response", lines=6, interactive=False)
        out_dual = gr.Textbox(label="Dual-layer watermarked response", lines=6, interactive=False)

    rep_plain = gr.Markdown()
    rep_l1 = gr.Markdown()
    rep_dual = gr.Markdown()

    btn.click(run_app, inputs=prompt_in,
              outputs=[out_plain, out_l1, out_dual, rep_plain, rep_l1, rep_dual])

    gr.Examples(
        examples=[
            ["The new vaccine trial results were published today and"],
            ["The central bank raised interest rates because"],
            ["The football team won the championship after"],
        ],
        inputs=prompt_in,
    )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75151372c12dcaf5e6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
